# Phase 3: Model Building (Isolation Forest)
### AI-Based Energy Anomaly Detector
**Objective**: Train an unsupervised Isolation Forest model on historical training features and detect abnormal energy consumption patterns.

---
### Why Isolation Forest?
- **Unsupervised**: Household electricity datasets do not come with ground truth anomaly labels.
- **Multi-dimensional Isolation**: Isolates anomalies by randomly selecting a feature and randomly splitting value; abnormal points require fewer recursive splits to isolate.
- **Fast & Explainable**: Scales efficiently across large time series and provides continuous decision scores.

In [ ]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

proj_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(proj_dir / 'src') not in sys.path:
    sys.path.append(str(proj_dir / 'src'))

from model import train_isolation_forest, evaluate_anomalies, FEATURE_COLS

## 1. Load Processed Datasets

In [ ]:
train_path = proj_dir / 'data' / 'processed' / 'train_features.csv'
test_path = proj_dir / 'data' / 'processed' / 'test_features.csv'

train_df = pd.read_csv(train_path, index_col=0, parse_dates=True)
test_df = pd.read_csv(test_path, index_col=0, parse_dates=True)

print(f"Train set: {train_df.shape} ({train_df.index.min()} to {train_df.index.max()})")
print(f"Test set: {test_df.shape} ({test_df.index.min()} to {test_df.index.max()})")
print(f"Selected features: {FEATURE_COLS}")

## 2. Train Isolation Forest
We train on `train_features` with `n_estimators=200` and `contamination=0.02` (~2% expected abnormal hours).

In [ ]:
model_path = proj_dir / 'src' / 'model.pkl'
model = train_isolation_forest(
    train_df=train_df,
    features=FEATURE_COLS,
    n_estimators=200,
    contamination=0.02,
    random_state=42,
    model_output_path=model_path
)
print("Model training and serialization complete.")

## 3. Predict & Score on Test Set
- `predictions`: `-1` = anomaly, `1` = normal
- `scores`: `decision_function()` — lower values correspond to higher abnormality.

In [ ]:
results_df, metrics = evaluate_anomalies(model, test_df, features=FEATURE_COLS)

# Save predictions
preds_path = proj_dir / 'outputs' / 'test_anomalies.csv'
results_df.to_csv(preds_path)
print(f"Detailed predictions written to: {preds_path}")

## 4. Anomaly Distribution & Visualization
Inspect the score distribution and sample flagged anomalies.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), dpi=120)

# Distribution of decision function scores
ax1.hist(results_df.loc[~results_df['is_anomaly'], 'anomaly_score'], bins=50, color='#22c55e', alpha=0.7, label='Normal (1)')
ax1.hist(results_df.loc[results_df['is_anomaly'], 'anomaly_score'], bins=30, color='#ef4444', alpha=0.8, label='Anomaly (-1)')
ax1.set_title('Distribution of Anomaly Decision Scores', fontweight='bold')
ax1.set_xlabel('Decision Function Score (Lower = More Anomalous)')
ax1.set_ylabel('Frequency')
ax1.legend()

# Scatter plot: Usage vs 24h Rolling Mean
ax2.scatter(results_df.loc[~results_df['is_anomaly'], 'rolling_mean_24h'], 
            results_df.loc[~results_df['is_anomaly'], 'usage'], 
            c='#94a3b8', alpha=0.4, s=12, label='Normal')
ax2.scatter(results_df.loc[results_df['is_anomaly'], 'rolling_mean_24h'], 
            results_df.loc[results_df['is_anomaly'], 'usage'], 
            c='#dc2626', alpha=0.9, s=28, label='Flagged Anomaly')
ax2.set_title('Hourly Usage vs. 24h Rolling Baseline', fontweight='bold')
ax2.set_xlabel('24h Rolling Baseline (kW)')
ax2.set_ylabel('Hourly Usage (kW)')
ax2.legend()

plt.tight_layout()
plt.savefig(proj_dir / 'outputs' / 'plots' / 'anomaly_score_distribution.png', dpi=300)
plt.show()